In [ ]:
# Applied Data Science Project 1: Motor Vehicle Crashes in NYC

## Maggie Boles, Adina Kaplan, Logan McCorkle, & Chana Ochs 

## 03/28/2026

### The purpose of this code is to combine multiple cleaned datasets for the Data Scientist, creating the model
    Source 1 (NYC Motor Vehicle Crashes): https://catalog.data.gov/dataset/motor-vehicle-collisions-crashes
    Source 2 (Traffic Signal and Stop Requests): https://data.cityofnewyork.us/Transportation/Traffic-Signal-and-All-Way-Stop-Study-Requests/w76s-c5u4/about_data 
    Source 3 (Traffic Signal Condition): https://data.cityofnewyork.us/Social-Services/Traffic-Signal-Condition_4/e5mv-wy8f/about_data
    Source 4 (NYS DOT Maintained Traffic Signals): https://data.gis.ny.gov/datasets/c61a41873d4f4b06bae79c23ad98b9c6_0/about

#### Throughout this code, we tried multiple times to have 4-way and 2-way stops recognized, but were unable to separate them. Instead, we went with a signalized, stop-controlled, and an uncontrolled approach, running multiple iterations to get this to work, but it was beyond my scope of ability. So this is the result. 

In [1]:
import pandas as pd
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")

# Load all files
crashes   = pd.read_csv('nyc_crashes_cleaned_2022_to_2026.csv')
signals   = pd.read_csv('NYSDOT_Traffic_Signals_cleaned.csv')
conditions= pd.read_csv('Traffic_Signal_Condition_cleaned.csv')
requests  = pd.read_csv('traffic_signal_requests_cleaned_with_2263.csv')

# 1. STANDARDIZE COORDINATE COLUMNS
print("Standardizing coordinate column names...")

for df in [crashes, signals, conditions, requests]:
    if 'X_STATEPLANE_2263' in df.columns:
        df.rename(columns={'X_STATEPLANE_2263': 'X_2263', 'Y_STATEPLANE_2263': 'Y_2263'}, inplace=True)
    elif 'X_2263' not in df.columns and 'X_STATEPLANE' in df.columns:
        df.rename(columns={'X_STATEPLANE': 'X_2263', 'Y_STATEPLANE': 'Y_2263'}, inplace=True)

# 2. CREATE MASTER INTERSECTIONS 
print("Creating master intersections...")

# Start with official NYSDOT signals
master = signals.copy()
master['INTERSECTION_SOURCE'] = 'NYSDOT_Signal'
master['HAS_SIGNAL'] = True

# Create GeoDataFrames
signals_gdf = gpd.GeoDataFrame(
    master,
    geometry=gpd.points_from_xy(master['X_2263'], master['Y_2263']),
    crs="EPSG:2263"
)

requests_gdf = gpd.GeoDataFrame(
    requests,
    geometry=gpd.points_from_xy(requests['X_2263'], requests['Y_2263']),
    crs="EPSG:2263"
)

# Keep only requests that are more than 50 ft from existing signals
near_signals = requests_gdf.sjoin_nearest(
    signals_gdf,
    how='left',
    max_distance=50,
    distance_col='DIST_TO_SIGNAL'
)

unique_requests = requests_gdf[near_signals['index_right'].isna()].copy()
unique_requests['INTERSECTION_SOURCE'] = 'Request'
unique_requests['HAS_SIGNAL'] = False

master = pd.concat([master, unique_requests], ignore_index=True)
print(f"Master intersections: {len(master):,} rows")

# 3. CREATE 50ft BUFFERS 
master_gdf = gpd.GeoDataFrame(
    master,
    geometry=gpd.points_from_xy(master['X_2263'], master['Y_2263']),
    crs="EPSG:2263"
)
master_gdf['buffer_50ft'] = master_gdf.geometry.buffer(50)

# 4. COUNT CRASHES WITHIN 50 FT 
crashes_gdf = gpd.GeoDataFrame(
    crashes,
    geometry=gpd.points_from_xy(crashes['X_2263'], crashes['Y_2263']),
    crs="EPSG:2263"
)

print("Performing spatial join for crashes (50 ft radius)...")
crashes_joined = gpd.sjoin_nearest(
    crashes_gdf,
    master_gdf,
    how='inner',
    max_distance=50,
    distance_col='DIST_TO_INTERSECTION'
)

crash_counts = crashes_joined.groupby('index_right').size().rename('CRASH_COUNT_50FT')

master_gdf = master_gdf.join(crash_counts, how='left')
master_gdf['CRASH_COUNT_50FT'] = master_gdf['CRASH_COUNT_50FT'].fillna(0).astype(int)

# 5. ADD CONTROL TYPE & FLAGS 
master_gdf['CONTROL_TYPE'] = 'Unknown'
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# Basic logic for stop-controlled from requests
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop', na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', na=False), 'CONTROL_TYPE'] = '2-Way Stop'

print(f"Final intersections: {len(master_gdf):,}")
print(f"Intersections with crashes: {(master_gdf['CRASH_COUNT_50FT'] > 0).sum():,}")

# 6. SAVE FINAL DATASET =
final_df = master_gdf.drop(columns=['geometry', 'buffer_50ft'], errors='ignore')

final_df.to_csv('MASTER_INTERSECTIONS_50ft.csv', index=False)

print("\n✅ MASTER_INTERSECTIONS_50ft.csv saved successfully!")
print(final_df[['CRASH_COUNT_50FT', 'HAS_SIGNAL', 'CONTROL_TYPE']].value_counts().head(10))

Loading datasets...
Standardizing coordinate column names...
Creating master intersections...
Master intersections: 41,958 rows
Performing spatial join for crashes (50 ft radius)...
Final intersections: 41,958
Intersections with crashes: 25,945

✅ MASTER_INTERSECTIONS_50ft.csv saved successfully!
CRASH_COUNT_50FT  HAS_SIGNAL  CONTROL_TYPE
0                 True        Signalized      11272
                  False       Unknown          4741
1                 False       Unknown          4160
2                 False       Unknown          3288
3                 False       Unknown          2853
4                 False       Unknown          2321
5                 False       Unknown          1754
6                 False       Unknown          1431
7                 False       Unknown          1063
8                 False       Unknown           974
Name: count, dtype: int64


#### First iteration of running and creating the master document that will combine our datasets to be used for the model later. 

In [2]:
# IMPROVED CONTROL TYPE LOGIC 
print("Improving CONTROL_TYPE logic...")

master_gdf['CONTROL_TYPE'] = 'Unknown'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Use RequestType where available
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# 3. Fallback using Problem or RequestType from other columns if needed
if 'PROBLEM' in master_gdf.columns:
    master_gdf.loc[master_gdf['PROBLEM'].str.contains('Stop', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

#ADD MORE USEFUL COLUMNS
master_gdf['HAS_CRASH'] = master_gdf['CRASH_COUNT_50FT'] > 0

# ... similar sjoin_nearest logic

print("\nFinal distribution:")
print(master_gdf[['CRASH_COUNT_50FT', 'HAS_SIGNAL', 'CONTROL_TYPE']].value_counts().head(15))

# SAVE FINAL CLEAN VERSION 
final_columns = [
    'SignalID', 'LocationDescription', 'MainRoute', 'IntersectingRoute1',
    'X_2263', 'Y_2263', 'BOROUGH',
    'HAS_SIGNAL', 'CONTROL_TYPE', 'CRASH_COUNT_50FT', 'HAS_CRASH',
    'INTERSECTION_SOURCE', 'RequestType', 'StatusDescription'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df.to_csv('MASTER_INTERSECTIONS_50ft.csv', index=False)

print("\n✅ Final file saved as MASTER_INTERSECTIONS_50ft.csv")
print(f"Total rows: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

Improving CONTROL_TYPE logic...

Final distribution:
CRASH_COUNT_50FT  HAS_SIGNAL  CONTROL_TYPE   
0                 True        Signalized         11272
                  False       Stop-Controlled     2686
1                 False       Unknown             2096
                              Stop-Controlled     2064
0                 False       Unknown             2055
2                 False       Unknown             1979
3                 False       Unknown             1947
4                 False       Unknown             1699
5                 False       Unknown             1351
2                 False       Stop-Controlled     1309
6                 False       Unknown             1191
7                 False       Unknown              922
3                 False       Stop-Controlled      906
8                 False       Unknown              891
9                 False       Unknown              763
Name: count, dtype: int64

✅ Final file saved as MASTER_INTERSECTIONS_50ft.c

In [3]:
# FINAL POLISH & CLEANUP 
print("Applying final improvements...")

# Ensure we have the columns we need
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'Unknown'))

# Better CONTROL_TYPE logic
master_gdf['CONTROL_TYPE'] = master_gdf.get('CONTROL_TYPE', 'Unknown')

# Prioritize official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# Improve stop-controlled detection from RequestType
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# Create a clean unique INTERSECTION_ID (more robust)
master_gdf['INTERSECTION_ID'] = (
    master_gdf['BOROUGH'].fillna('UNKNOWN').astype(str) + "_" +
    master_gdf.get('MainRoute', master_gdf.get('MainStreet', '')).fillna('').astype(str) + "_" +
    master_gdf.get('IntersectingRoute1', 
                   master_gdf.get('CrossStreet1', 
                                  master_gdf.get('CrossStreet_1', ''))).fillna('').astype(str)
).str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)

# Add binary crash flag
master_gdf['HAS_CRASH'] = master_gdf['CRASH_COUNT_50FT'] > 0

#  FINAL COLUMN SELECTION
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription', 'MainRoute', 'IntersectingRoute1',
    'HAS_SIGNAL', 'CONTROL_TYPE',
    'CRASH_COUNT_50FT', 'HAS_CRASH',
    'INTERSECTION_SOURCE', 'RequestType', 'StatusDescription'
]

# Keep only columns that actually exist
final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

# Sort by most interesting first
final_df = final_df.sort_values(by=['CRASH_COUNT_50FT', 'HAS_SIGNAL'], ascending=[False, False])

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ Final polished dataset saved as: MASTER_INTERSECTIONS_50ft_FINAL.csv")
print(f"Total intersections: {len(final_df):,}")
print(f"With at least one crash: {final_df['HAS_CRASH'].sum():,}")

print("\nCONTROL_TYPE distribution:")
print(final_df['CONTROL_TYPE'].value_counts())

print("\nSample of final data:")
print(final_df.head(8)[['INTERSECTION_ID', 'CONTROL_TYPE', 'CRASH_COUNT_50FT', 'HAS_SIGNAL']])

Applying final improvements...

✅ Final polished dataset saved as: MASTER_INTERSECTIONS_50ft_FINAL.csv
Total intersections: 41,958
With at least one crash: 25,945

CONTROL_TYPE distribution:
CONTROL_TYPE
Unknown            22006
Signalized         11277
Stop-Controlled     8675
Name: count, dtype: int64

Sample of final data:
      INTERSECTION_ID CONTROL_TYPE  CRASH_COUNT_50FT  HAS_SIGNAL
11366         BRONX__      Unknown               165       False
12226         BRONX__      Unknown               165       False
12351         BRONX__      Unknown               165       False
12992         BRONX__      Unknown               165       False
14035         BRONX__      Unknown               165       False
15938         BRONX__      Unknown               165       False
11299      BROOKLYN__      Unknown               135       False
12058      BROOKLYN__      Unknown               135       False


#### Attempting to find the 2-way and 4-way stops. 

In [ ]:
print("Creating clean final master dataset...")

# Ensure key columns exist with fallbacks
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'UNKNOWN')).fillna('UNKNOWN')
master_gdf['MAIN_ROUTE'] = master_gdf.get('MainRoute', master_gdf.get('MainStreet', '')).fillna('')
master_gdf['CROSS_ROUTE'] = (
    master_gdf.get('IntersectingRoute1', '') .fillna('') + 
    master_gdf.get('CrossStreet1', '') .fillna('')
).str.strip()

# Better INTERSECTION_ID
master_gdf['INTERSECTION_ID'] = (
    master_gdf['BOROUGH'].astype(str) + "_" +
    master_gdf['MAIN_ROUTE'].astype(str) + "_" +
    master_gdf['CROSS_ROUTE'].astype(str)
).str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)

# Improved CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled / Other'

master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# Binary flags
master_gdf['HAS_CRASH'] = master_gdf['CRASH_COUNT_50FT'] > 0

# ====================== FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription', 'MAIN_ROUTE', 'CROSS_ROUTE',
    'HAS_SIGNAL', 'CONTROL_TYPE',
    'CRASH_COUNT_50FT', 'HAS_CRASH',
    'INTERSECTION_SOURCE', 'RequestType', 'StatusDescription'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

# Sort: highest crash count first, then signals
final_df = final_df.sort_values(by=['CRASH_COUNT_50FT', 'HAS_SIGNAL'], ascending=[False, False])

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET SAVED: MASTER_INTERSECTIONS_50ft_FINAL.csv")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nCONTROL_TYPE distribution:")
print(final_df['CONTROL_TYPE'].value_counts())

print("\nSample of clean data:")
print(final_df.head(10)[['INTERSECTION_ID', 'CONTROL_TYPE', 'CRASH_COUNT_50FT', 'HAS_SIGNAL', 'BOROUGH']])

#### Still not getting the control_type to pull out the intersections. 

In [5]:
# LAST CLEAN-UP PASS
print("Final clean-up pass...")

# Clean up INTERSECTION_ID (make it more readable)
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace('__', ' @ ', regex=False)
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace('_', ' ', regex=False).str.strip()

# Add a few more useful summary columns
master_gdf['CRASH_PER_YEAR'] = master_gdf['CRASH_COUNT_50FT'] / 4.25   # approx 4.25 years of data (2022-Mar 2026)

# Final column selection - clean and focused
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'CRASH_PER_YEAR',
    'HAS_CRASH',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

# Sort: highest crash locations first
final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL CLEAN DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 'CRASH_COUNT_50FT']])

print("\nCONTROL_TYPE summary:")
print(final_df['CONTROL_TYPE'].value_counts())



Final clean-up pass...

✅ FINAL CLEAN DATASET SAVED
Total intersections: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
                       INTERSECTION_ID   BOROUGH          CONTROL_TYPE  \
12351  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
11366        BRONX @ WEST FORDHAM ROAD     BRONX  Uncontrolled / Other   
14035  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
15938  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
12992  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
12226  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
37162       BROOKLYN @ EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   
15817       BROOKLYN @ EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   
12058        BROOKLYN @ BUFFALO AVENUE  BROOKLYN  Uncontrolled / Other   
15103       BROOKLYN @ EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   

       CRASH_COUNT_50FT  
12351               1

In [6]:
print("Creating final delivery-ready version...")

# Improve labels for major roads / highways
master_gdf['CONTROL_TYPE'] = master_gdf['CONTROL_TYPE'].replace('Uncontrolled / Other', 'Uncontrolled / Other')

# Better labeling for expressways and major roads
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    main_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[main_col].str.contains('EXPRESSWAY|DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway / Ramp'

# Final INTERSECTION_ID cleanup
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace(' @ ', ' @ ', regex=False)
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Final columns for delivery
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'HAS_CRASH',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY.csv', index=False)

print("\n✅ READY FOR DELIVERY FILE SAVED: MASTER_INTERSECTIONS_50ft_READY.csv")
print(f"Total rows: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 'CRASH_COUNT_50FT']])

print("\nCONTROL_TYPE final summary:")
print(final_df['CONTROL_TYPE'].value_counts())

Creating final delivery-ready version...

✅ READY FOR DELIVERY FILE SAVED: MASTER_INTERSECTIONS_50ft_READY.csv
Total rows: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
                       INTERSECTION_ID   BOROUGH          CONTROL_TYPE  \
12351  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
11366        BRONX @ WEST FORDHAM ROAD     BRONX  Uncontrolled / Other   
14035  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
15938  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
12992  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
12226  BRONX @ MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
37162       BROOKLYN @ EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   
15817       BROOKLYN @ EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   
12058        BROOKLYN @ BUFFALO AVENUE  BROOKLYN  Uncontrolled / Other   
15103       BROOKLYN @ EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other  

In [7]:
print("Creating final delivery-ready version with improved labeling...")

# Ensure borough exists
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'UNKNOWN')).fillna('UNKNOWN')

# Improve CONTROL_TYPE for high-crash locations
master_gdf['CONTROL_TYPE'] = master_gdf.get('CONTROL_TYPE', 'Uncontrolled / Other')

# Official signals take priority
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# Better logic for expressways, ramps, and major roads
main_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
if main_col in master_gdf.columns:
    master_gdf.loc[master_gdf[main_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway / Ramp / Major Arterial'

# Stop-controlled from requests
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# Clean up INTERSECTION_ID for readability
master_gdf['INTERSECTION_ID'] = (
    master_gdf['BOROUGH'].astype(str) + " @ " +
    master_gdf.get('MainRoute', master_gdf.get('MAIN_ROUTE', '')).fillna('') + " / " +
    master_gdf.get('IntersectingRoute1', master_gdf.get('CrossStreet1', '')).fillna('')
).str.strip().str.replace(r'\s+', ' ', regex=True)

# Final useful columns
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'HAS_CRASH',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DELIVERY FILE SAVED: MASTER_INTERSECTIONS_50ft_FINAL.csv")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 'CRASH_COUNT_50FT']])

print("\nCONTROL_TYPE final summary:")
print(final_df['CONTROL_TYPE'].value_counts())

Creating final delivery-ready version with improved labeling...

✅ FINAL DELIVERY FILE SAVED: MASTER_INTERSECTIONS_50ft_FINAL.csv
Total intersections: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
  INTERSECTION_ID   BOROUGH          CONTROL_TYPE  CRASH_COUNT_50FT
0       BRONX @ /     BRONX  Uncontrolled / Other               165
1       BRONX @ /     BRONX  Uncontrolled / Other               165
2       BRONX @ /     BRONX  Uncontrolled / Other               165
3       BRONX @ /     BRONX  Uncontrolled / Other               165
4       BRONX @ /     BRONX  Uncontrolled / Other               165
5       BRONX @ /     BRONX  Uncontrolled / Other               165
6    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135
7    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135
8    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135
9    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135

CONTROL_TYPE final summary:
CONTROL

In [8]:
print("Creating final delivery-ready version with 311 complaint counts...")

# Ensure borough and route columns exist
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'UNKNOWN')).fillna('UNKNOWN')
main_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'

# ====================== IMPROVED CONTROL_TYPE ======================
master_gdf['CONTROL_TYPE'] = 'Uncontrolled / Other'

master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

if main_col in master_gdf.columns:
    master_gdf.loc[master_gdf[main_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway / Ramp / Major Arterial'

if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# ====================== COUNT 311 COMPLAINTS WITHIN 50 FT ======================
print("Adding 311 signal complaint counts within 50 ft...")

conditions_gdf = gpd.GeoDataFrame(
    conditions,
    geometry=gpd.points_from_xy(conditions['X_2263'], conditions['Y_2263']),
    crs="EPSG:2263"
)

# Spatial join: count conditions within 50 ft of each intersection
conditions_joined = gpd.sjoin_nearest(
    conditions_gdf,
    master_gdf,
    how='inner',
    max_distance=50,
    distance_col='DIST_TO_INTERSECTION'
)

complaint_counts = conditions_joined.groupby('index_right').size().rename('COMPLAINT_COUNT_50FT')

master_gdf = master_gdf.join(complaint_counts, how='left')
master_gdf['COMPLAINT_COUNT_50FT'] = master_gdf['COMPLAINT_COUNT_50FT'].fillna(0).astype(int)

# ====================== FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'HAS_CRASH',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

# Clean up INTERSECTION_ID for readability
final_df['INTERSECTION_ID'] = final_df['INTERSECTION_ID'].str.replace('__', ' @ ', regex=False)
final_df['INTERSECTION_ID'] = final_df['INTERSECTION_ID'].str.replace(r'\s+', ' ', regex=True).str.strip()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET WITH 311 COMPLAINTS SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nCONTROL_TYPE summary:")
print(final_df['CONTROL_TYPE'].value_counts())

Creating final delivery-ready version with 311 complaint counts...
Adding 311 signal complaint counts within 50 ft...

✅ FINAL DATASET WITH 311 COMPLAINTS SAVED
Total intersections: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
  INTERSECTION_ID   BOROUGH          CONTROL_TYPE  CRASH_COUNT_50FT  \
0       BRONX @ /     BRONX  Uncontrolled / Other               165   
1       BRONX @ /     BRONX  Uncontrolled / Other               165   
2       BRONX @ /     BRONX  Uncontrolled / Other               165   
3       BRONX @ /     BRONX  Uncontrolled / Other               165   
4       BRONX @ /     BRONX  Uncontrolled / Other               165   
5       BRONX @ /     BRONX  Uncontrolled / Other               165   
6    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   
7    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   
8    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   
9    BROOKLYN @ /  BROOKLYN  Uncontrolled /

In [9]:
print("Creating clean, delivery-ready master intersections file...")

# Fix borough and route columns with robust fallbacks
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'UNKNOWN')).fillna('UNKNOWN')

# Route columns with multiple possible names
main_route = master_gdf.get('MainRoute', master_gdf.get('MAIN_ROUTE', master_gdf.get('MainStreet', ''))).fillna('')
cross_route = master_gdf.get('IntersectingRoute1', 
                    master_gdf.get('CrossStreet1', 
                    master_gdf.get('CrossStreet_1', ''))).fillna('')

# Create clean INTERSECTION_ID
master_gdf['INTERSECTION_ID'] = (
    master_gdf['BOROUGH'].astype(str) + " @ " +
    main_route.astype(str) + " / " +
    cross_route.astype(str)
).str.strip().str.replace(r'\s+', ' ', regex=True).str.replace(' / ', ' @ ', regex=False)

# ====================== IMPROVED CONTROL_TYPE ======================
master_gdf['CONTROL_TYPE'] = 'Uncontrolled / Other'

# Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# Expressways and major roads
master_gdf.loc[master_gdf[main_route.name if hasattr(main_route, 'name') else 'MainRoute']
               .str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
               'CONTROL_TYPE'] = 'Expressway / Ramp / Major Arterial'

# Stop-controlled from requests
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# ====================== FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'HAS_CRASH'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DELIVERY-READY FILE SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nCONTROL_TYPE summary:")
print(final_df['CONTROL_TYPE'].value_counts())

Creating clean, delivery-ready master intersections file...

✅ FINAL DELIVERY-READY FILE SAVED
Total intersections: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
  INTERSECTION_ID   BOROUGH          CONTROL_TYPE  CRASH_COUNT_50FT  \
0       BRONX @ /     BRONX  Uncontrolled / Other               165   
1       BRONX @ /     BRONX  Uncontrolled / Other               165   
2       BRONX @ /     BRONX  Uncontrolled / Other               165   
3       BRONX @ /     BRONX  Uncontrolled / Other               165   
4       BRONX @ /     BRONX  Uncontrolled / Other               165   
5       BRONX @ /     BRONX  Uncontrolled / Other               165   
6    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   
7    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   
8    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   
9    BROOKLYN @ /  BROOKLYN  Uncontrolled / Other               135   

   COMPLAINT_COUNT_50FT  
0          

In [10]:
print("Creating clean, delivery-ready master intersections file (final version)...")

# Robust column fallbacks
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'UNKNOWN')).fillna('UNKNOWN')

# Route columns with multiple possible names
main_route = (
    master_gdf.get('MainRoute', '').
    fillna(master_gdf.get('MAIN_ROUTE', '')).
    fillna(master_gdf.get('MainStreet', ''))
).fillna('')

cross_route = (
    master_gdf.get('IntersectingRoute1', '').
    fillna(master_gdf.get('CrossStreet1', '')).
    fillna(master_gdf.get('CrossStreet_1', ''))
).fillna('')

# Create better INTERSECTION_ID
master_gdf['INTERSECTION_ID'] = (
    master_gdf['BOROUGH'].astype(str) + " @ " +
    main_route.astype(str) + " / " +
    cross_route.astype(str)
).str.strip().str.replace(r'\s+', ' ', regex=True)

# If still empty, use LocationDescription as fallback
master_gdf.loc[master_gdf['INTERSECTION_ID'].str.endswith(' / '), 'INTERSECTION_ID'] = (
    master_gdf['BOROUGH'] + " @ " + master_gdf['LocationDescription'].fillna('Unknown Location')
)

# ====================== FINAL CONTROL_TYPE ======================
master_gdf['CONTROL_TYPE'] = 'Uncontrolled / Other'

master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# Expressway / major road detection
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway / Ramp / Major Arterial'

if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# ====================== FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'HAS_CRASH',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DELIVERY-READY FILE SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nCONTROL_TYPE summary:")
print(final_df['CONTROL_TYPE'].value_counts())

Creating clean, delivery-ready master intersections file (final version)...

✅ FINAL DELIVERY-READY FILE SAVED
Total intersections: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
                     INTERSECTION_ID   BOROUGH          CONTROL_TYPE  \
0  BRONX @ / MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
1        BRONX @ / WEST FORDHAM ROAD     BRONX  Uncontrolled / Other   
2  BRONX @ / MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
3  BRONX @ / MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
4  BRONX @ / MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
5  BRONX @ / MAJOR DEEGAN EXPRESSWAY     BRONX  Uncontrolled / Other   
6       BROOKLYN @ / EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   
7       BROOKLYN @ / EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   
8        BROOKLYN @ / BUFFALO AVENUE  BROOKLYN  Uncontrolled / Other   
9       BROOKLYN @ / EASTERN PARKWAY  BROOKLYN  Uncontrolled / Other   

   CRASH_C

#### In the last few iterations, it shows uncontrolled, signalized, stop-controlled, and expressway, but we still cannot get the data Adina wanted. 

In [11]:
print("Creating clean, delivery-ready master intersections file...")

# Robust column handling
master_gdf['BOROUGH'] = master_gdf.get('BOROUGH', master_gdf.get('Borough', 'UNKNOWN')).fillna('UNKNOWN')

# Try multiple possible route column names
main_route = master_gdf.get('MainRoute', 
                   master_gdf.get('MAIN_ROUTE', 
                   master_gdf.get('MainStreet', ''))).fillna('')

cross_route = master_gdf.get('IntersectingRoute1', 
                    master_gdf.get('CrossStreet1', 
                    master_gdf.get('CrossStreet_1', ''))).fillna('')

# Build INTERSECTION_ID with fallback
master_gdf['INTERSECTION_ID'] = (
    master_gdf['BOROUGH'].astype(str) + " @ " +
    main_route.astype(str)
)

# Add cross street only if it exists and is meaningful
has_cross = cross_route.str.strip() != ''
master_gdf.loc[has_cross, 'INTERSECTION_ID'] = (
    master_gdf.loc[has_cross, 'INTERSECTION_ID'] + " / " + cross_route.loc[has_cross]
)

# Final fallback: use LocationDescription if ID is still poor
poor_id = master_gdf['INTERSECTION_ID'].str.contains('@ $|@ /$', regex=True) | (master_gdf['INTERSECTION_ID'].str.len() < 10)
master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].astype(str) + " @ " + 
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# CONTROL_TYPE (final touch) 
master_gdf['CONTROL_TYPE'] = 'Uncontrolled / Other'
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway / Ramp / Major Arterial'

if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = '4-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = '2-Way Stop'
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop-Controlled'

# FINAL COLUMNS =
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'HAS_CRASH',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DELIVERY-READY FILE SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nCONTROL_TYPE summary:")
print(final_df['CONTROL_TYPE'].value_counts())

Creating clean, delivery-ready master intersections file...

✅ FINAL DELIVERY-READY FILE SAVED
Total intersections: 41,958
With crashes: 25,945

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH          CONTROL_TYPE  \
0     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
1     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
2     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
3     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
4     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
5     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
6  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
7  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
8  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
9  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   

   CRASH_COUNT_50FT  COMPLAINT_COUNT_50FT  
0               165                    17  
1   

In [12]:
print("Applying final requested improvements...")

# 1. Deduplication: Keep only one row per physical location
# Use INTERSECTION_ID + coordinates for deduplication
master_gdf = master_gdf.drop_duplicates(
    subset=['INTERSECTION_ID', 'X_2263', 'Y_2263'], 
    keep='first'
).reset_index(drop=True)

print(f"After deduplication: {len(master_gdf):,} unique intersections")

# 2. Binary conversion (1/0 instead of True/False)
master_gdf['HAS_SIGNAL'] = master_gdf['HAS_SIGNAL'].astype(int)
master_gdf['HAS_CRASH']   = master_gdf['HAS_CRASH'].astype(int)

# 3. Improved Stop-Controlled logic (try to distinguish 2-way vs 4-way)
master_gdf['CONTROL_TYPE'] = master_gdf['CONTROL_TYPE'].replace('Stop-Controlled', 'Stop-Controlled (Unknown)')

if 'RequestType' in master_gdf.columns:
    # 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 
                   'CONTROL_TYPE'] = '4-Way Stop'
    
    # 2-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 
                   'CONTROL_TYPE'] = '2-Way Stop'

# Final cleanup of INTERSECTION_ID (make it nicer)
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace(' @ / ', ' @ ', regex=False)
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.strip()

# FINAL COLUMNS FOR DELIVERY 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CONTROL_TYPE',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'INTERSECTION_SOURCE'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

# Sort by crash count descending
final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DELIVERY FILE READY!")
print(f"Total unique intersections after deduplication: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'CONTROL_TYPE', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT', 
                         'HAS_SIGNAL', 'HAS_CRASH']])

print("\nCONTROL_TYPE final summary:")
print(final_df['CONTROL_TYPE'].value_counts())

print("\nHAS_SIGNAL distribution:")
print(final_df['HAS_SIGNAL'].value_counts())

Applying final requested improvements...
After deduplication: 27,253 unique intersections

✅ FINAL DELIVERY FILE READY!
Total unique intersections after deduplication: 27,253
With crashes: 14,150

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH          CONTROL_TYPE  \
0     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
1  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
2  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
3     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
4  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
5     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
6  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
7     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   
8  BROOKLYN @ Unknown Location  BROOKLYN  Uncontrolled / Other   
9     BRONX @ Unknown Location     BRONX  Uncontrolled / Other   

   CRASH_COUNT_50FT  COMPLAINT_COUNT_50F

#### Now we have the 4-way stops but still missing 2 way stops. 

In [13]:
print("Creating clean final dataset with proper one-hot encoding...")

# Reset and clean CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway detection
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway_Ramp'

# 3. Stop sign logic (prioritize 2-Way and 4-Way)
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

# ====================== SAFE ONE-HOT ENCODING ======================
# Create one-hot with explicit categories to avoid duplicates
categories = ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'], prefix='', prefix_sep='')

# Ensure all 4 columns exist (fill missing with 0)
for cat in categories:
    if cat not in one_hot.columns:
        one_hot[cat] = 0

one_hot = one_hot[categories]   # Keep only the 4 we want

# Rename to exact names your teammate requested
one_hot.columns = ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']

# Add to master_gdf
master_gdf = pd.concat([master_gdf, one_hot], axis=1)

print("Safe one-hot encoding completed.")

# ====================== FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL CLEAN DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating clean final dataset with proper one-hot encoding...
Safe one-hot encoding completed.

✅ FINAL CLEAN DATASET SAVED
Total intersections: 27,253
With crashes: 14,150

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX       False      False          0   
1  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
2  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
3     BRONX @ Unknown Location     BRONX       False      False          0   
4  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
5     BRONX @ Unknown Location     BRONX       False      False          0   
6  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
7     BRONX @ Unknown Location     BRONX       False      False          0   
8  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
9     BRONX

#### 2-Way stops are present, but not properly identified. 

In [14]:
print("Final clean one-hot encoding fix...")

# Reset CONTROL_TYPE to base categories
master_gdf['CONTROL_TYPE'] = master_gdf['CONTROL_TYPE'].map({
    'Signalized': 'Signalized',
    'Stop_4Way': 'Stop_4Way',
    'Stop_2Way': 'Stop_2Way',
    'Uncontrolled': 'Uncontrolled',
    'Expressway / Ramp': 'Uncontrolled',   # treat as uncontrolled for simplicity
    'Expressway_Ramp': 'Uncontrolled'
}).fillna('Uncontrolled')

# Safe one-hot encoding - create from scratch
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

# Force exactly the 4 columns your teammate wants
for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

# Keep only the 4 columns and rename if needed
one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']]

# Add to master_gdf (drop any previous one-hot columns to avoid duplicates)
master_gdf = master_gdf.loc[:, ~master_gdf.columns.str.contains('Signalized|Stop_|Uncontrolled')]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

print("Clean one-hot encoding completed with exactly 4 columns.")

# FINAL COLUMNS =
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL CLEAN DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Final clean one-hot encoding fix...
Clean one-hot encoding completed with exactly 4 columns.

✅ FINAL CLEAN DATASET SAVED
Total intersections: 27,253
With crashes: 14,150

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX       False      False          0   
1  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
2  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
3     BRONX @ Unknown Location     BRONX       False      False          0   
4  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
5     BRONX @ Unknown Location     BRONX       False      False          0   
6  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
7     BRONX @ Unknown Location     BRONX       False      False          0   
8  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
9     BRONX 

In [15]:
print("Final clean one-hot encoding fix...")

# Reset CONTROL_TYPE to base categories
master_gdf['CONTROL_TYPE'] = master_gdf['CONTROL_TYPE'].map({
    'Signalized': 'Signalized',
    'Stop_4Way': 'Stop_4Way',
    'Stop_2Way': 'Stop_2Way',
    'Uncontrolled': 'Uncontrolled',
    'Expressway / Ramp': 'Uncontrolled',   # treat as uncontrolled for simplicity
    'Expressway_Ramp': 'Uncontrolled'
}).fillna('Uncontrolled')

# Safe one-hot encoding - create from scratch
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

# Force exactly the 4 columns your teammate wants
for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

# Keep only the 4 columns and rename if needed
one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']]

# Add to master_gdf (drop any previous one-hot columns to avoid duplicates)
master_gdf = master_gdf.loc[:, ~master_gdf.columns.str.contains('Signalized|Stop_|Uncontrolled')]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

print("Clean one-hot encoding completed with exactly 4 columns.")

# FINAL COLUMNS 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL CLEAN DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Final clean one-hot encoding fix...
Clean one-hot encoding completed with exactly 4 columns.

✅ FINAL CLEAN DATASET SAVED
Total intersections: 27,253
With crashes: 14,150

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX       False      False          0   
1  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
2  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
3     BRONX @ Unknown Location     BRONX       False      False          0   
4  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
5     BRONX @ Unknown Location     BRONX       False      False          0   
6  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
7     BRONX @ Unknown Location     BRONX       False      False          0   
8  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
9     BRONX 

In [16]:
print("Creating final polished delivery file...")

# Final cleanup of INTERSECTION_ID
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace(' @ Unknown Location', ' @ Unknown Location', regex=False)

# Use LocationDescription as a better fallback when street info is missing
mask = master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False)
master_gdf.loc[mask, 'INTERSECTION_ID'] = (
    master_gdf.loc[mask, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[mask, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Final one-hot columns are already correct from previous step

# Final columns for delivery
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL POLISHED DATASET READY FOR YOUR TEAMMATE")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating final polished delivery file...

✅ FINAL POLISHED DATASET READY FOR YOUR TEAMMATE
Total intersections: 27,253
With crashes: 14,150

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX       False      False          0   
1  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
2  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
3     BRONX @ Unknown Location     BRONX       False      False          0   
4  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
5     BRONX @ Unknown Location     BRONX       False      False          0   
6  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
7     BRONX @ Unknown Location     BRONX       False      False          0   
8  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
9     BRONX @ Unknown Location     BRONX   

In [17]:
print("Creating the final polished version for your teammate...")

# Final INTERSECTION_ID cleanup with better fallback
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].fillna('')

# Use LocationDescription when street info is missing or poor
poor_id_mask = (master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | 
                (master_gdf['INTERSECTION_ID'].str.len() < 15))

master_gdf.loc[poor_id_mask, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id_mask, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id_mask, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Make sure one-hot columns are present and clean
one_hot_cols = ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']
for col in one_hot_cols:
    if col not in master_gdf.columns:
        master_gdf[col] = 0

# Final columns for delivery
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL POLISHED DATASET READY TO SEND")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating the final polished version for your teammate...

✅ FINAL POLISHED DATASET READY TO SEND
Total intersections: 27,253
With crashes: 14,150

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX       False      False          0   
1  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
2  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
3     BRONX @ Unknown Location     BRONX       False      False          0   
4  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
5     BRONX @ Unknown Location     BRONX       False      False          0   
6  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
7     BRONX @ Unknown Location     BRONX       False      False          0   
8  BROOKLYN @ Unknown Location  BROOKLYN       False      False          0   
9     BRONX @ Unknown Location     BR

In [18]:
print("Applying teammate's final feedback...")

#  1. BETTER STOP_2WAY LABELING
# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# Expressway / Major roads
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway_Ramp'

# Improved stop detection from RequestType
if 'RequestType' in master_gdf.columns:
    # 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # 2-Way Stop (this is the key part)
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Any remaining stop-related requests default to Stop_2Way
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count after improved logic: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")

# ====================== 2. CLEAN ONE-HOT ENCODING (All 0/1) ======================
# Create one-hot from the cleaned CONTROL_TYPE
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

# Force exactly the 4 columns
for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']]

# Convert everything to integer 0/1 (no True/False)
one_hot = one_hot.astype(int)

# Add to master_gdf (drop any previous one-hot columns to avoid duplicates)
cols_to_drop = [col for col in master_gdf.columns if col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']]
master_gdf = master_gdf.drop(columns=cols_to_drop, errors='ignore')

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

print("Clean one-hot encoding (all 0/1) completed.")

# ====================== 3. FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET READY FOR YOUR TEAMMATE")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary (0/1):")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Applying teammate's final feedback...
Stop_2Way count after improved logic: 0
Clean one-hot encoding (all 0/1) completed.

✅ FINAL DATASET READY FOR YOUR TEAMMATE
Total intersections: 27,253
With crashes: 14,150
Stop_2Way count: 0

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0          0   
8  BROOKLYN @ Unknown Location

In [19]:
print("Final aggressive fix for Stop_2Way labeling...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway detection
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway_Ramp'

# 3. Aggressive stop detection (this is the key change)
if 'RequestType' in master_gdf.columns:
    # Explicit 4-Way
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Explicit 2-Way
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Any remaining stop-related (Flasher, Beacon, Stop Sign, etc.) → treat as Stop_2Way
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|Stop Sign', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

# Final fallback: any remaining "Stop-Controlled" or vague stop entries → Stop_2Way
master_gdf.loc[master_gdf['CONTROL_TYPE'].isin(['Stop-Controlled', 'Stop-Controlled (Unknown)']), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count after aggressive labeling: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")

# CLEAN ONE-HOT (0/1 only) 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

# Force exactly the 4 columns
for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Drop any old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# ====================== FINAL OUTPUT ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET WITH IMPROVED STOP_2WAY LABELING")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Final aggressive fix for Stop_2Way labeling...
Stop_2Way count after aggressive labeling: 0

✅ FINAL DATASET WITH IMPROVED STOP_2WAY LABELING
Total intersections: 27,253
With crashes: 14,150
Stop_2Way count: 0

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0          0   
8  BROOKLYN @ Unknown Location  BROOKLYN           

In [20]:
print("Final aggressive Stop_2Way labeling (using source as fallback)...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official NYSDOT signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Major roads
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway_Ramp'

# 3. Aggressive stop labeling
# Any intersection that came from the Requests dataset AND is not a signal = Stop_2Way
master_gdf.loc[(master_gdf['INTERSECTION_SOURCE'] == 'Request') & 
               (master_gdf['HAS_SIGNAL'] == False), 'CONTROL_TYPE'] = 'Stop_2Way'

# Any remaining vague stop-related entries
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|All-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count after source-based labeling: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")

# CLEAN ONE-HOT ENCODING 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove any old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL OUTPUT
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET WITH IMPROVED STOP_2WAY LABELING")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Final aggressive Stop_2Way labeling (using source as fallback)...
Stop_2Way count after source-based labeling: 17,610

✅ FINAL DATASET WITH IMPROVED STOP_2WAY LABELING
Total intersections: 27,253
With crashes: 14,150
Stop_2Way count: 17,610

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknow

#### Attempting to get the 2-way stops again, and now it is lumping the 4-way, 2-way, and Uncontrolled together. This is not what we want. 

In [21]:
print("Creating the final polished version for delivery...")

# Improve INTERSECTION_ID readability
master_gdf['INTERSECTION_ID'] = master_gdf['INTERSECTION_ID'].str.replace(' @ Unknown Location', ' @ Unknown Location')

# Use LocationDescription when street info is missing
poor_id = master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | (master_gdf['INTERSECTION_ID'].str.len() < 20)
master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Final columns
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DELIVERY-READY FILE CREATED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating the final polished version for delivery...

✅ FINAL DELIVERY-READY FILE CREATED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 17,610

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
9     BRONX @ Unknown Locat

In [22]:
print("Creating the absolute final version for your teammate...")

# Improve INTERSECTION_ID with better fallback
master_gdf['INTERSECTION_ID'] = master_gdf.get('INTERSECTION_ID', '').fillna('')

poor_id = (master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | 
           (master_gdf['INTERSECTION_ID'].str.len() < 20))

master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Final columns (clean and focused)
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ THIS IS THE FINAL VERSION READY TO SEND")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating the absolute final version for your teammate...

✅ THIS IS THE FINAL VERSION READY TO SEND
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 17,610

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
9     BRONX @ Un

In [23]:
print("Creating improved 2-Way and 4-Way stop labeling...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Major roads
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Expressway_Ramp'

# 3. Improved stop labeling
if 'RequestType' in master_gdf.columns:
    # Explicit 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Explicit 2-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Any other stop-related request (Flasher, Beacon, Stop Sign, etc.) → treat as Stop_2Way
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|Stop Sign|Stop-Controlled', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

# Final fallback: Any intersection from the requests dataset that is not a signal is likely a stop-controlled location
master_gdf.loc[(master_gdf['INTERSECTION_SOURCE'] == 'Request') & 
               (master_gdf['HAS_SIGNAL'] == False) & 
               (master_gdf['CONTROL_TYPE'] == 'Uncontrolled'), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")

# ====================== CLEAN ONE-HOT ENCODING ======================
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns if they exist
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# ====================== FINAL OUTPUT ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET WITH IMPROVED 2-WAY / 4-WAY LABELING")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating improved 2-Way and 4-Way stop labeling...
Stop_2Way count: 11,631
Stop_4Way count: 5,979

✅ FINAL DATASET WITH IMPROVED 2-WAY / 4-WAY LABELING
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 11,631
Stop_4Way: 5,979

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknown Loc

#### Here we were able to get the 4-way stop and 2-way stops, but now we have lost the uncontrolled, which means the 2-way absorbed them, and it isn't separating the two properly. 

In [24]:
print("Creating the final polished version for your teammate...")

# Final INTERSECTION_ID cleanup with better fallback
master_gdf['INTERSECTION_ID'] = master_gdf.get('INTERSECTION_ID', '').fillna('')

poor_id = (master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | 
           (master_gdf['INTERSECTION_ID'].str.len() < 20))

master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Final columns
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL POLISHED DATASET READY TO SEND")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating the final polished version for your teammate...

✅ FINAL POLISHED DATASET READY TO SEND
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 11,631
Stop_4Way: 5,979

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
9 

#### Here we were able to get the 4-way stop and 2-way stops, but now we have lost the uncontrolled, which means the 2-way absorbed them, and it isn't separating the two properly. 

In [25]:
print("Creating the absolute final polished version...")

# Final INTERSECTION_ID improvement
master_gdf['INTERSECTION_ID'] = master_gdf.get('INTERSECTION_ID', '').fillna('')

# Use LocationDescription when street info is missing or poor
poor_id = (master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | 
           (master_gdf['INTERSECTION_ID'].str.len() < 20))

master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Final columns for delivery
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ THIS IS THE FINAL VERSION TO SEND TO YOUR TEAMMATE")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating the absolute final polished version...

✅ THIS IS THE FINAL VERSION TO SEND TO YOUR TEAMMATE
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 11,631
Stop_4Way: 5,979

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1 

#### Here we were able to get the 4-way stop and 2-way stops, but now we have lost the uncontrolled, which means the 2-way absorbed them, and it isn't separating the two properly. 

In [26]:
print("Applying teammate's requested fixes (conservative Stop_2Way labeling)...")

# Reset CONTROL_TYPE to a clean base
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official NYSDOT signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp / Major Arterial (should be Uncontrolled)
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Conservative Stop labeling (only clear stop-controlled cases)
if 'RequestType' in master_gdf.columns:
    # Explicit 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Explicit 2-Way Stop + other clear stop signs
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way|Stop Sign|Stop-Controlled', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count after conservative labeling: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")

# ====================== CLEAN ONE-HOT ENCODING (0/1 only) ======================
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

# Force exactly the 4 columns your teammate wants
for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove any old one-hot columns to avoid duplicates
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# ====================== FINAL COLUMNS ======================
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL DATASET WITH CORRECTED STOP LABELING")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary (0/1):")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Applying teammate's requested fixes (conservative Stop_2Way labeling)...
Stop_2Way count after conservative labeling: 0
Stop_4Way count: 5,979

✅ FINAL DATASET WITH CORRECTED STOP LABELING
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 0
Stop_4Way: 5,979

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0        

#### Here we were able to get the 4-way stop and lost the 2way into the uncontrolled. 

In [27]:
print("Balanced labeling for Stop_2Way and Uncontrolled...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp / Major Arterial → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Balanced stop detection
if 'RequestType' in master_gdf.columns:
    # Explicit 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Explicit 2-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Other stop-related terms → Stop_2Way (this is the balanced part)
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|Stop Sign|Stop-Controlled', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Uncontrolled count: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

#  CLEAN ONE-HOT ENCODING 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove any old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL OUTPUT 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL BALANCED DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Balanced labeling for Stop_2Way and Uncontrolled...
Stop_2Way count: 0
Stop_4Way count: 5,979
Uncontrolled count: 11,697

✅ FINAL BALANCED DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 0
Stop_4Way: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0          0   
8  BROOK

In [28]:
print("Simple & practical labeling for modeling...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Simple rule: Any intersection from the Requests dataset that is NOT a signal = Stop_2Way
master_gdf.loc[(master_gdf['INTERSECTION_SOURCE'] == 'Request') & 
               (master_gdf['HAS_SIGNAL'] == False), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Uncontrolled count: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT ENCODING (0/1) 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns if they exist
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL COLUMNS 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL PRACTICAL DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Simple & practical labeling for modeling...
Stop_2Way count: 17,610
Stop_4Way count: 0
Uncontrolled count: 66

✅ FINAL PRACTICAL DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 17,610
Stop_4Way: 0
Uncontrolled: 66

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
7     BRONX @ Unknown Location     BRONX           0          0          1   
8  BROOKLYN @ Unknown

#### Still not able to get the 4-way and 2-way stops, and now the uncontrolled are being absorbed in the Uncontrolled. 

In [29]:
print("Creating balanced Stop_2Way / Uncontrolled split...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Balanced stop labeling
if 'RequestType' in master_gdf.columns:
    # Clear 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Clear 2-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Other stop-related terms (Flasher, Beacon, Stop Sign, etc.) → Stop_2Way
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|Stop Sign', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

# Final fallback: Any remaining vague stop entries stay as Uncontrolled (conservative)
# (We do NOT move everything from requests into Stop_2Way)

print(f"Stop_2Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Uncontrolled count: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT (0/1)
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL OUTPUT 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL BALANCED DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Creating balanced Stop_2Way / Uncontrolled split...
Stop_2Way count: 0
Stop_4Way count: 5,979
Uncontrolled count: 11,697

✅ FINAL BALANCED DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 0
Stop_4Way: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0          0   
8  BROOK

#### Now the 4way and Uncontrolled are separated out. 

In [30]:
print("Final balanced labeling - conservative Stop_2Way...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Conservative stop labeling
if 'RequestType' in master_gdf.columns:
    # Clear 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Clear 2-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Other stop-related terms
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|Stop Sign', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Uncontrolled count: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT (0/1)
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL OUTPUT 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL BALANCED DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Final balanced labeling - conservative Stop_2Way...
Stop_2Way count: 0
Stop_4Way count: 5,979
Uncontrolled count: 11,697

✅ FINAL BALANCED DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 0
Stop_4Way: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0          0   
8  BROOK

In [31]:
print("Final balanced labeling - conservative Stop_2Way...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Conservative stop labeling
if 'RequestType' in master_gdf.columns:
    # Clear 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Clear 2-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'
    
    # Other stop-related terms
    master_gdf.loc[(master_gdf['CONTROL_TYPE'] == 'Uncontrolled') & 
                   master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|Stop Sign', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Stop_2Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Stop_4Way count: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Uncontrolled count: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT (0/1) 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL OUTPUT 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL BALANCED DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Final balanced labeling - conservative Stop_2Way...
Stop_2Way count: 0
Stop_4Way count: 5,979
Uncontrolled count: 11,697

✅ FINAL BALANCED DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 0
Stop_4Way: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          0          0   
8  BROOK

In [32]:
print("Simple practical labeling for modeling...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Keep the existing Stop_4Way from previous runs (if any)
if 'Stop_4Way' in master_gdf.columns:
    master_gdf.loc[master_gdf['Stop_4Way'] == 1, 'CONTROL_TYPE'] = 'Stop_4Way'

# No forced Stop_2Way - leave as Uncontrolled unless clear 4-Way

print(f"Signalized: {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_4Way: {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Stop_2Way: 0 (not enough data to distinguish)")
print(f"Uncontrolled: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT (0/1)
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL OUTPUT
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)

print("\n✅ FINAL PRACTICAL DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_2Way: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way: {final_df['Stop_4Way'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

Simple practical labeling for modeling...
Signalized: 9,577
Stop_4Way: 5,979
Stop_2Way: 0 (not enough data to distinguish)
Uncontrolled: 11,697

✅ FINAL PRACTICAL DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_2Way: 0
Stop_4Way: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
7     BRONX @ Unknown Location     BRONX           0          

In [33]:
    print("Final simplified labeling for modeling (Stop_Controlled category)...")
    
    # Reset CONTROL_TYPE
    master_gdf['CONTROL_TYPE'] = 'Uncontrolled'
    
    # 1. Official signals
    master_gdf.loc[master_gdf['HAS_SIGNAL'] == True, 'CONTROL_TYPE'] = 'Signalized'
    
    # 2. Expressway / Ramp → Uncontrolled
    if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
        route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
        master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                       'CONTROL_TYPE'] = 'Uncontrolled'
    
    # 3. Stop_Controlled = any stop-related request (combining 2-Way and 4-Way)
    if 'RequestType' in master_gdf.columns:
        master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|All-Way|4-Way|2-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_Controlled'
    
    print(f"Signalized: {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
    print(f"Stop_Controlled: {(master_gdf['CONTROL_TYPE'] == 'Stop_Controlled').sum():,}")
    print(f"Uncontrolled: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")
    
    # CLEAN ONE-HOT (0/1) 
    one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])
    
    for col in ['Signalized', 'Stop_Controlled', 'Uncontrolled']:
        if col not in one_hot.columns:
            one_hot[col] = 0
    
    one_hot = one_hot[['Signalized', 'Stop_Controlled', 'Uncontrolled']].astype(int)
    
    # Remove old one-hot columns
    master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_Controlled', 'Uncontrolled'])]
    
    master_gdf = pd.concat([master_gdf, one_hot], axis=1)
    
    #  FINAL COLUMNS 
    final_columns = [
        'INTERSECTION_ID',
        'X_2263', 'Y_2263',
        'BOROUGH',
        'LocationDescription',
        'HAS_SIGNAL',
        'HAS_CRASH',
        'CRASH_COUNT_50FT',
        'COMPLAINT_COUNT_50FT',
        'Signalized',
        'Stop_Controlled',
        'Uncontrolled'
    ]
    
    final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()
    
    final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)
    
    final_df.to_csv('MASTER_INTERSECTIONS_50ft_FINAL.csv', index=False)
    
    print("\n✅ FINAL SIMPLIFIED DATASET SAVED")
    print(f"Total intersections: {len(final_df):,}")
    print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
    print(f"Stop_Controlled: {final_df['Stop_Controlled'].sum():,}")
    print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")
    
    print("\nTop 10 highest crash intersections:")
    print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_Controlled', 'Uncontrolled', 
                             'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])
    
    print("\nOne-hot CONTROL_TYPE summary:")
    print(final_df[['Signalized', 'Stop_Controlled', 'Uncontrolled']].sum())

Final simplified labeling for modeling (Stop_Controlled category)...
Signalized: 9,577
Stop_Controlled: 5,979
Uncontrolled: 11,697

✅ FINAL SIMPLIFIED DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_Controlled: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_Controlled  \
0     BRONX @ Unknown Location     BRONX           0                0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
3     BRONX @ Unknown Location     BRONX           0                0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
5     BRONX @ Unknown Location     BRONX           0                0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
7     BRONX @ Unknown Location     BRONX           0                0   
8  BROOKLYN @ Unknown Location  BROOKLYN        

In [34]:
print("Creating Stop_2Way using best available proxy...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == 1, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp / Major Arterial → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Stop labeling using best proxy
if 'RequestType' in master_gdf.columns:
    # Clear 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Stop_2Way = any stop-related term OR any request that is not a signal
    stop_related = master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|2-Way', case=False, na=False)
    master_gdf.loc[stop_related, 'CONTROL_TYPE'] = 'Stop_2Way'

# Final proxy: Any intersection that came from the Requests dataset and is not a signal = Stop_2Way
master_gdf.loc[(master_gdf.get('INTERSECTION_SOURCE', '') == 'Request') & 
               (master_gdf['HAS_SIGNAL'] == 0) & 
               (master_gdf['CONTROL_TYPE'] == 'Uncontrolled'), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Signalized:   {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_4Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Stop_2Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Uncontrolled: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# ====================== CLEAN ONE-HOT ======================
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# Save the result
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv', index=False)

print("\n✅ FILE SAVED: MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv")
print(f"Total intersections: {len(final_df):,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")

print("\nOne-hot summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

Creating Stop_2Way using best available proxy...
Signalized:   9,577
Stop_4Way:    0
Stop_2Way:    17,610
Uncontrolled: 66

✅ FILE SAVED: MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv
Total intersections: 27,253
Stop_2Way count: 17,610

One-hot summary:
Signalized       9577
Stop_4Way           0
Stop_2Way       17610
Uncontrolled       66
dtype: int64

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          1   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
3     BRONX @ Unknown Location     BRONX           0          0          1   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          1   
5     BRONX @ Unknown Location     BRONX           0          0          1   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          

In [35]:
print("Creating more accurate 2-Way Stop separation...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == 1, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp / Major Arterial → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Conservative stop labeling
if 'RequestType' in master_gdf.columns:
    # Clear 4-Way Stop
    master_gdf.loc[master_gdf['RequestType'].str.contains('All-Way Stop|4-Way', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # Clear 2-Way Stop - only when we see explicit stop terms
    master_gdf.loc[master_gdf['RequestType'].str.contains('2-Way|Stop Sign|Flasher|Beacon', case=False, na=False), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Signalized:   {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_4Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Stop_2Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Uncontrolled: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# ====================== CLEAN ONE-HOT ======================
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# Save
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv', index=False)

print("\n✅ FILE SAVED: MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv")
print(f"Total intersections: {len(final_df):,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")

print("\nOne-hot summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

Creating more accurate 2-Way Stop separation...
Signalized:   9,577
Stop_4Way:    5,979
Stop_2Way:    0
Uncontrolled: 11,697

✅ FILE SAVED: MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv
Total intersections: 27,253
Stop_2Way count: 0

One-hot summary:
Signalized       9577
Stop_4Way        5979
Stop_2Way           0
Uncontrolled    11697
dtype: int64

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0          0  

In [36]:
print("Final 3-category version (Signalized, Stop_Controlled, Uncontrolled)...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == 1, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Stop_Controlled = any request that mentions stop-related terms
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|All-Way|4-Way|2-Way', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_Controlled'

print(f"Signalized:      {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_Controlled: {(master_gdf['CONTROL_TYPE'] == 'Stop_Controlled').sum():,}")
print(f"Uncontrolled:    {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT (0/1) 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_Controlled', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_Controlled', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_Controlled', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# IMPROVE INTERSECTION_ID 
master_gdf['INTERSECTION_ID'] = master_gdf.get('INTERSECTION_ID', '').fillna('')

poor_id = (master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | 
           (master_gdf['INTERSECTION_ID'].str.len() < 20))

master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# FINAL SAVE 
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_Controlled',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv', index=False)

print("\n✅ FINAL 3-CATEGORY FILE SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_Controlled: {final_df['Stop_Controlled'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_Controlled', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_Controlled', 'Uncontrolled']].sum())

Final 3-category version (Signalized, Stop_Controlled, Uncontrolled)...
Signalized:      9,577
Stop_Controlled: 5,979
Uncontrolled:    11,697

✅ FINAL 3-CATEGORY FILE SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_Controlled: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_Controlled  \
0     BRONX @ Unknown Location     BRONX           0                0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
3     BRONX @ Unknown Location     BRONX           0                0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
5     BRONX @ Unknown Location     BRONX           0                0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
7     BRONX @ Unknown Location     BRONX           0                0   
8  BROOKLYN @ Unknown Location  BROOKLYN

#### I wanted to keep the signalized, Stop-controlled, and Uncontrolled. It was the neat, tidy way to get the data we needed for the model, after all the iterations failed and didn't compile as they should. 

In [37]:
print("Splitting Stop_Controlled into Stop_4Way and Stop_2Way based on classmate's suggestion...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == 1, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Stop labeling using previous Stop_Controlled logic
if 'RequestType' in master_gdf.columns:
    # First identify all stop-related requests
    stop_related = master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|All-Way|4-Way|2-Way', case=False, na=False)
    
    # ALL-WAY STOP → Stop_4Way
    master_gdf.loc[master_gdf['RequestType'] == 'ALL-WAY STOP', 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # All other stop-related (but not ALL-WAY STOP) → Stop_2Way
    master_gdf.loc[(stop_related) & (master_gdf['RequestType'] != 'ALL-WAY STOP') & 
                   (master_gdf['CONTROL_TYPE'] == 'Uncontrolled'), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Signalized:   {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_4Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Stop_2Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Uncontrolled: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT 
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# Save final file
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv', index=False)

print("\n✅ FINAL FILE SAVED WITH Stop_2Way SEPARATED")
print(f"Total intersections: {len(final_df):,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way count: {final_df['Stop_4Way'].sum():,}")

print("\nOne-hot summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

Splitting Stop_Controlled into Stop_4Way and Stop_2Way based on classmate's suggestion...
Signalized:   9,577
Stop_4Way:    5,979
Stop_2Way:    0
Uncontrolled: 11,697

✅ FINAL FILE SAVED WITH Stop_2Way SEPARATED
Total intersections: 27,253
Stop_2Way count: 0
Stop_4Way count: 5,979

One-hot summary:
Signalized       9577
Stop_4Way        5979
Stop_2Way           0
Uncontrolled    11697
dtype: int64

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Un

In [38]:
print("Splitting Stop_Controlled into Stop_4Way and Stop_2Way based on classmate's suggestion...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == 1, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Stop labeling using previous Stop_Controlled logic
if 'RequestType' in master_gdf.columns:
    # First identify all stop-related requests
    stop_related = master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|All-Way|4-Way|2-Way', case=False, na=False)
    
    # ALL-WAY STOP → Stop_4Way
    master_gdf.loc[master_gdf['RequestType'] == 'ALL-WAY STOP', 'CONTROL_TYPE'] = 'Stop_4Way'
    
    # All other stop-related (but not ALL-WAY STOP) → Stop_2Way
    master_gdf.loc[(stop_related) & (master_gdf['RequestType'] != 'ALL-WAY STOP') & 
                   (master_gdf['CONTROL_TYPE'] == 'Uncontrolled'), 'CONTROL_TYPE'] = 'Stop_2Way'

print(f"Signalized:   {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_4Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_4Way').sum():,}")
print(f"Stop_2Way:    {(master_gdf['CONTROL_TYPE'] == 'Stop_2Way').sum():,}")
print(f"Uncontrolled: {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].astype(int)

# Remove old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# Save final file
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_4Way',
    'Stop_2Way',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv', index=False)

print("\n✅ FINAL FILE SAVED WITH Stop_2Way SEPARATED")
print(f"Total intersections: {len(final_df):,}")
print(f"Stop_2Way count: {final_df['Stop_2Way'].sum():,}")
print(f"Stop_4Way count: {final_df['Stop_4Way'].sum():,}")

print("\nOne-hot summary:")
print(final_df[['Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled']].sum())

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_4Way', 'Stop_2Way', 'Uncontrolled', 
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

Splitting Stop_Controlled into Stop_4Way and Stop_2Way based on classmate's suggestion...
Signalized:   9,577
Stop_4Way:    5,979
Stop_2Way:    0
Uncontrolled: 11,697

✅ FINAL FILE SAVED WITH Stop_2Way SEPARATED
Total intersections: 27,253
Stop_2Way count: 0
Stop_4Way count: 5,979

One-hot summary:
Signalized       9577
Stop_4Way        5979
Stop_2Way           0
Uncontrolled    11697
dtype: int64

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_4Way  Stop_2Way  \
0     BRONX @ Unknown Location     BRONX           0          0          0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
3     BRONX @ Unknown Location     BRONX           0          0          0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0          0          0   
5     BRONX @ Unknown Location     BRONX           0          0          0   
6  BROOKLYN @ Un

In [39]:
print("Final 3-Category Model-Ready Dataset (Signalized / Stop_Controlled / Uncontrolled)...")

# Reset CONTROL_TYPE
master_gdf['CONTROL_TYPE'] = 'Uncontrolled'

# 1. Official NYSDOT signals
master_gdf.loc[master_gdf['HAS_SIGNAL'] == 1, 'CONTROL_TYPE'] = 'Signalized'

# 2. Expressway / Ramp / Major Arterial → Uncontrolled
if 'MainRoute' in master_gdf.columns or 'MAIN_ROUTE' in master_gdf.columns:
    route_col = 'MainRoute' if 'MainRoute' in master_gdf.columns else 'MAIN_ROUTE'
    master_gdf.loc[master_gdf[route_col].str.contains('DEEGAN|GOWANUS|BROOKLYN-QUEENS|MAJOR DEEGAN|EXPRESSWAY|PKWY|BOULEVARD', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Uncontrolled'

# 3. Stop_Controlled = any stop-related request
if 'RequestType' in master_gdf.columns:
    master_gdf.loc[master_gdf['RequestType'].str.contains('Stop|Flasher|Beacon|All-Way|4-Way|2-Way', case=False, na=False), 
                   'CONTROL_TYPE'] = 'Stop_Controlled'

print(f"Signalized:      {(master_gdf['CONTROL_TYPE'] == 'Signalized').sum():,}")
print(f"Stop_Controlled: {(master_gdf['CONTROL_TYPE'] == 'Stop_Controlled').sum():,}")
print(f"Uncontrolled:    {(master_gdf['CONTROL_TYPE'] == 'Uncontrolled').sum():,}")

# CLEAN ONE-HOT ENCODING
one_hot = pd.get_dummies(master_gdf['CONTROL_TYPE'])

for col in ['Signalized', 'Stop_Controlled', 'Uncontrolled']:
    if col not in one_hot.columns:
        one_hot[col] = 0

one_hot = one_hot[['Signalized', 'Stop_Controlled', 'Uncontrolled']].astype(int)

# Remove any old one-hot columns
master_gdf = master_gdf.loc[:, ~master_gdf.columns.isin(['Signalized', 'Stop_Controlled', 'Uncontrolled'])]

master_gdf = pd.concat([master_gdf, one_hot], axis=1)

# FINAL POLISH
# Improve INTERSECTION_ID
master_gdf['INTERSECTION_ID'] = master_gdf.get('INTERSECTION_ID', '').fillna('')

poor_id = (master_gdf['INTERSECTION_ID'].str.contains('Unknown Location', na=False) | 
           (master_gdf['INTERSECTION_ID'].str.len() < 20))

master_gdf.loc[poor_id, 'INTERSECTION_ID'] = (
    master_gdf.loc[poor_id, 'BOROUGH'].fillna('UNKNOWN').astype(str) + " @ " +
    master_gdf.loc[poor_id, 'LocationDescription'].fillna('Unknown Location').astype(str)
)

# Final columns
final_columns = [
    'INTERSECTION_ID',
    'X_2263', 'Y_2263',
    'BOROUGH',
    'LocationDescription',
    'HAS_SIGNAL',
    'HAS_CRASH',
    'CRASH_COUNT_50FT',
    'COMPLAINT_COUNT_50FT',
    'Signalized',
    'Stop_Controlled',
    'Uncontrolled'
]

final_df = master_gdf[[col for col in final_columns if col in master_gdf.columns]].copy()

final_df = final_df.sort_values(by='CRASH_COUNT_50FT', ascending=False).reset_index(drop=True)

final_df.to_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv', index=False)

print("\n✅ FINAL 3-CATEGORY DATASET SAVED")
print(f"Total intersections: {len(final_df):,}")
print(f"With crashes: {final_df['HAS_CRASH'].sum():,}")
print(f"Stop_Controlled: {final_df['Stop_Controlled'].sum():,}")
print(f"Uncontrolled: {final_df['Uncontrolled'].sum():,}")

print("\nTop 10 highest crash intersections:")
print(final_df.head(10)[['INTERSECTION_ID', 'BOROUGH', 'Signalized', 'Stop_Controlled', 'Uncontrolled',
                         'CRASH_COUNT_50FT', 'COMPLAINT_COUNT_50FT']])

print("\nOne-hot CONTROL_TYPE summary:")
print(final_df[['Signalized', 'Stop_Controlled', 'Uncontrolled']].sum())

Final 3-Category Model-Ready Dataset (Signalized / Stop_Controlled / Uncontrolled)...
Signalized:      9,577
Stop_Controlled: 5,979
Uncontrolled:    11,697

✅ FINAL 3-CATEGORY DATASET SAVED
Total intersections: 27,253
With crashes: 14,150
Stop_Controlled: 5,979
Uncontrolled: 11,697

Top 10 highest crash intersections:
               INTERSECTION_ID   BOROUGH  Signalized  Stop_Controlled  \
0     BRONX @ Unknown Location     BRONX           0                0   
1  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
2  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
3     BRONX @ Unknown Location     BRONX           0                0   
4  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
5     BRONX @ Unknown Location     BRONX           0                0   
6  BROOKLYN @ Unknown Location  BROOKLYN           0                0   
7     BRONX @ Unknown Location     BRONX           0                0   
8  BROOKLYN @ Unknown L

#### Finally settled on the One-Hot Encoding for Signalized, Stop-Controlled (No specificity), and Uncontrolled intersections. This was our final decision, which made the most sense for our model and project, given the problems encountered during one-hot encoding. 

In [40]:
# Burough Count for Presentation
# Import Libraries
import pandas as pd

# Load your final master file
df = pd.read_csv('MASTER_INTERSECTIONS_50ft_READY_FOR_MODEL.csv')

print("=== CRASH COUNT BY BOROUGH ===\n")

# Total crashes by borough
borough_crashes = df.groupby('BOROUGH')['CRASH_COUNT_50FT'].sum().reset_index()
borough_crashes = borough_crashes.sort_values(by='CRASH_COUNT_50FT', ascending=False)

print("Total Crashes by Borough:")
print(borough_crashes)

# Also show number of intersections and average crashes per intersection
summary = df.groupby('BOROUGH').agg(
    Total_Intersections=('BOROUGH', 'count'),
    Total_Crashes=('CRASH_COUNT_50FT', 'sum'),
    Avg_Crashes_per_Intersection=('CRASH_COUNT_50FT', 'mean'),
    Intersections_with_Crashes=('HAS_CRASH', 'sum')
).round(2)

summary = summary.sort_values(by='Total_Crashes', ascending=False)
print("\nDetailed Summary by Borough:")
print(summary)

# Optional: Percentage of total crashes
total_crashes = df['CRASH_COUNT_50FT'].sum()
borough_crashes['Percentage'] = (borough_crashes['CRASH_COUNT_50FT'] / total_crashes * 100).round(2)
print("\nCrashes by Borough with Percentage:")
print(borough_crashes)

=== CRASH COUNT BY BOROUGH ===

Total Crashes by Borough:
         BOROUGH  CRASH_COUNT_50FT
1       BROOKLYN             34151
3         QUEENS             22755
2      MANHATTAN             15606
0          BRONX             11548
4  STATEN ISLAND              4498
5        UNKNOWN                18

Detailed Summary by Borough:
               Total_Intersections  Total_Crashes  \
BOROUGH                                             
BROOKLYN                      5295          34151   
QUEENS                        6366          22755   
MANHATTAN                     1725          15606   
BRONX                         2102          11548   
STATEN ISLAND                 2122           4498   
UNKNOWN                       9643             18   

               Avg_Crashes_per_Intersection  Intersections_with_Crashes  
BOROUGH                                                                  
BROOKLYN                               6.45                        4751  
QUEENS              

#### Quick rundown by borough to be used by Presenter Chana Ochs to ensure we have counts by borough for her portion. 